# S4 on a Colab GPU: Labarca et al. (2024) reproduction on RetailRocket

1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > Run all.
3. The last cell downloads `nip_results.zip`; upload it to the project thread (or unzip it into `results/nip/` of the repo).

One seed of the full grid (2 backbones x 4 heads, up to 50 epochs, early stopping) takes about an hour; all 5 seeds about 4 to 5 hours.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/samyakbansal5-eng/rl-seqrec
%cd rl-seqrec
!pip -q install -r requirements.txt

In [ ]:
!bash data/download.sh && python data/preprocess_retailrocket.py

In [ ]:
SEEDS = [0]          # paper: [0, 1, 2, 3, 4]
EPOCHS, PATIENCE = 50, 3
import subprocess, itertools, os
os.makedirs('results/nip/logs', exist_ok=True)
for seed, head, backbone in itertools.product(SEEDS, ['none', 'sqn', 'eval', 'hist'], ['sasrec', 'gru']):
    log = f'results/nip/logs/{backbone}-{head}-s{seed}.log'
    print('running', backbone, head, seed, flush=True)
    with open(log, 'w') as f:
        subprocess.run(['python', 'scripts/train_nip.py', '--backbone', backbone, '--head', head, '--seed', str(seed),
                        '--epochs', str(EPOCHS), '--patience', str(PATIENCE), '--device', 'auto'], stdout=f, stderr=subprocess.STDOUT, check=True)

In [ ]:
!python scripts/summarize.py && cat results/nip_table.md

In [ ]:
!zip -qr nip_results.zip results/nip results/nip_table.md
from google.colab import files
files.download('nip_results.zip')